# 01: Data Cleaning & Transformation
***
**Author:** Zain Zabidi
**Date:** February 2026

**Objective:** To clean raw Census CSVs, filter for Western Australian data, and apply midpoint mappings to create a consolidated numerical dataset for analysis.

## 1. Environment Setup & Data Loading
First we import the raw Census CSVs and the midpoint dictionaries.

**Datasets to be processed:**
* **G01 & G02:** Baseline population characteristics and official ABS medians for validation.
* **G33, G38 , G40:** Categorical data requiring midpoint mapping (Income, Mortgage, Rent).
* **G46, G54:** Count-based data (Labour Force, Industry).

### 1.1 Import Libraries and Define Paths
Using **Pandas** for transformation and **OS/JSON** for file management and dictionary retrieval.

In [206]:
import pandas as pd
import os
import json

# Set directory paths for raw data, dictionaries, and processed output
RAW_DATA_PATH = '../data/raw/'
DICT_PATH = '../data/dictionary/'
PROCESSED_DATA_PATH = '../data/processed/'

# Ensure processed directory exists
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# List all required ABS Census tables for the project
G_FILES = [
    'G01', 'G02', 'G33', 'G38', 'G40', 'G46A', 'G46B', 'G54A', 'G54B'
]

# Mapping dicts created in notebook 00 to respective data types
DICT_FILES = {
    'income': 'income_midpoints.json',
    'mortgage': 'mortgage_midpoints.json',
    'rent': 'rent_midpoints.json'
}

### 1.2 Data Ingestion
We load the CSV files into a dictionary of DataFrames.These datasets were acquired already pre-filtered for Western Australia.

In [207]:
# Dict to store the filtered DataFrames
dfs = {}

for g_file in G_FILES:
    # Construct file path
    file_name = f'2021Census_{g_file}_WA_SAL.csv'
    full_path = os.path.join(RAW_DATA_PATH, file_name)

    # Load dataset
    dfs[g_file] = pd.read_csv(full_path)

# Verify number of rows in samples
for name, df in dfs.items():
    print(f"{name}: {df.shape[0]} records")

G01: 1701 records
G02: 1701 records
G33: 1701 records
G38: 1701 records
G40: 1701 records
G46A: 1701 records
G46B: 1701 records
G54A: 1701 records
G54B: 1701 records


### 1.3 Load Mapping Dictionaries
We load the midpoint dictionaries previously exported to JSON. These will be used to transform categorical range data into continuous numerical values.

In [208]:
# Dict to store loaded midpoint mappings
midpoint_dicts = {}

for key, filename in DICT_FILES.items():
    dict_full_path = os.path.join(DICT_PATH, filename)

    with open(dict_full_path, 'r') as f:
        midpoint_dicts[key] = json.load(f)

# Check data loaded correctly
for key, mapping in midpoint_dicts.items():
    sample = dict(list(mapping.items())[:3])
    print(f"{key.capitalize()} mapping sample: {sample}")

Income mapping sample: {'Negative_Nil_income_Tot': 0, 'HI_1_149_Tot': 75.0, 'HI_150_299_Tot': 224.5}
Mortgage mapping sample: {'M_0_299_Tot': 149.5, 'M_300_449_Tot': 374.5, 'M_450_599_Tot': 524.5}
Rent mapping sample: {'R_1_74_Tot': 37.5, 'R_75_99_Tot': 87.0, 'R_100_149_Tot': 124.5}


## Section 2: Data Transformation
In this section, we apply the midpoint mappings to the Census tables. We will process each major variable (Rent, Mortgage, Income) individually to ensure accuracy before merging the features into a single dataset.

### 2.1 Process Weekly Rent (G40)
We apply the rent midpoint dictionary to the G40 table. This allows us to estimate the total weekly rent expenditure per suburb by multiplying household counts by bracket midpoints.

**Formula:**
$$Total\_Rent\_Weekly = \sum (Household\_Count_{range} \times Midpoint_{range})$$

*Where:*
* ***Household_Count:** Number of households reported in a specific rent range (e.g., $150-$199).*
* ***Midpoint:** The numerical representative value for that range (e.g., $174.5).*
* ***Total:** The sum of all rent paid by all reporting households in the suburb combined.*

In [209]:
# Identify columns in G40 that match to our rent midpoint keys
rent_cols = list(midpoint_dicts['rent'].keys())

# Create copy of G40 dataframe to alter
rent_df = dfs['G40'].copy()

# Calculate estimated total rent for each bracket
# Multiply household count by midpoint value
for col in rent_cols:
    midpoint = midpoint_dicts['rent'][col]
    rent_df[f'{col}_val'] = rent_df[col] * midpoint

# Sum calculated values for single 'total_rent_weekly' column
val_cols = [f'{col}_val' for col in rent_cols]
rent_df['total_rent_weekly'] = rent_df[val_cols].sum(axis=1)

# Display check for first few results
display(rent_df[['SAL_CODE_2021', 'total_rent_weekly']].head())

,SAL_CODE_2021,total_rent_weekly
0,SAL50001,0.0
1,SAL50002,38475.5
2,SAL50003,998.0
3,SAL50004,0.0
4,SAL50005,0.0


**Observation:**
The `total_rent_weekly` value represents the estimated aggregate rent for the entire suburb.
* For example, **SAL50002** shows ~$38,475, which is the sum of all rent paid by all households that fell within our defined price ranges.
* Suburbs with **0.0** (like SAL50001) indicate areas with no reported rental households in those specific Census categories (likely industrial areas, parks, or zero-population zones).

This aggregate figure serves as our **numerator** for calculating the mean weekly rent in the next step.

### 2.2 Calculate Renting Households
To calculate a meaningful average, we need to know how many households contributed to the total rent expenditure. We sum the counts from the same price ranges used in the previous step.

In [210]:
# Sum counts of households in relevant rent price ranges
rent_df['renting_households_count'] = rent_df[rent_cols].sum(axis=1)

# Calculate Average Weekly Rent per Suburb
# Avoid dividing by zero for no-renter suburbs with a small check
rent_df['avg_weekly_rent'] = (
    rent_df['total_rent_weekly'] / rent_df['renting_households_count']
).fillna(0)

# Display check
display(rent_df[['SAL_CODE_2021', 'total_rent_weekly', 'renting_households_count', 'avg_weekly_rent']].head())

,SAL_CODE_2021,total_rent_weekly,renting_households_count,avg_weekly_rent
0,SAL50001,0.0,0,0.000000
1,SAL50002,38475.5,104,369.956731
2,SAL50003,998.0,4,249.500000
3,SAL50004,0.0,0,0.000000
4,SAL50005,0.0,0,0.000000


### 2.3 Process Monthly Mortgage (G38)
Table G38 breaks down mortgage repayments by dwelling structure (House, Townhouse, etc.). We first aggregate these sub-categories into total columns for each price range, then apply our midpoint mappings to calculate the average monthly mortgage per suburb.

In [211]:
# Create copy of the G38 dataframe
mortgage_df = dfs['G38'].copy()

# Identify mortgage brackets from dict
mort_cols = list(midpoint_dicts['mortgage'].keys())

# Sum dwelling-specific columns to create aggregate Total columns
for target_col in mort_cols:
    base_range = target_col.replace('_Tot', '')
    sub_cols = [c for c in mortgage_df if c.startswith(base_range)]
    mortgage_df[target_col] = mortgage_df[sub_cols].sum(axis=1)

# Apply midpoints to calculate estimated expenditure for each price range
for col in mort_cols:
    midpoint = midpoint_dicts['mortgage'][col]
    mortgage_df[f'{col}_val'] = mortgage_df[col] * midpoint

# Calculate total monthly expenditure and mortgage-paying households count
val_cols_m = [f'{col}_val' for col in mort_cols]
mortgage_df['total_mortgage_monthly'] = mortgage_df[val_cols_m].sum(axis=1)
mortgage_df['mortgage_households_count'] = mortgage_df[mort_cols].sum(axis=1)

# Calculate average monthly mortgage per suburb
mortgage_df['avg_monthly_mortgage'] = (
    mortgage_df['total_mortgage_monthly'] / mortgage_df['mortgage_households_count']
).fillna(0)

# Display check
display(mortgage_df[['SAL_CODE_2021', 'total_mortgage_monthly', 'mortgage_households_count', 'avg_monthly_mortgage']].head())

,SAL_CODE_2021,total_mortgage_monthly,mortgage_households_count,avg_monthly_mortgage
0,SAL50001,54591.0,18,3032.833333
1,SAL50002,521859.5,288,1812.012153
2,SAL50003,32992.0,16,2062.000000
3,SAL50004,0.0,0,0.000000
4,SAL50005,0.0,0,0.000000


### 2.4 Process Weekly Household Income (G33)
We apply the income midpoint dictionary to the G33 tbale. Unlike the mortgage data, this table already provides pre-aggregated total columns for each income price range, allowing for a direct mapping to numerical values.

In [212]:
# Create copy of G33 dataframe
inc_df = dfs['G33'].copy()

# Identify income price range from dictionary
inc_cols = list(midpoint_dicts['income'].keys())

# Apply midpoints to calculate estimated income for each price range
for col in inc_cols:
    midpoint = midpoint_dicts['income'][col]
    inc_df[f'{col}_val'] = inc_df[col] * midpoint

# Calculate total weekly income & income-reporting households count
val_cols_i = [f'{col}_val' for col in inc_cols]
inc_df['total_income_weekly'] = inc_df[val_cols_i].sum(axis=1)
inc_df['income_households_count'] = inc_df[inc_cols].sum(axis=1)

# Calculate average weekly household income per suburb
inc_df['avg_weekly_income'] = (
    inc_df['total_income_weekly'] / inc_df['income_households_count']
).fillna(0)

# Display check
display(inc_df[['SAL_CODE_2021', 'total_income_weekly', 'income_households_count', 'avg_weekly_income']].head())

,SAL_CODE_2021,total_income_weekly,income_households_count,avg_weekly_income
0,SAL50001,41245.5,14,2946.107143
1,SAL50002,834671.0,456,1830.418860
2,SAL50003,38287.5,29,1320.258621
3,SAL50004,19200.0,4,4800.000000
4,SAL50005,1348.5,3,449.500000


### 2.5 Process Industry of Employment (G54A & G54B)
To address the research question regarding employment concentration, we extract industry totals from the G54 tables. Since male data is in G54A and female data is in G54B, we merge these tables and aggregate gendered counts to calculate total industry employment and the total workforce per suburb.

In [213]:
# Create copies of G54 tables
ind_a_df = dfs['G54A'].copy()
ind_b_df = dfs['G54B'].copy()

# Define specific industry columns for extraction
industry_targets = {
    'Mining': {'M': 'M_Mining_Tot', 'F': 'F_Mining_Tot'},
    'Education': {'M': 'M_Educ_trng_Tot', 'F': 'F_Educ_trng_Tot'},
    'Healthcare': {'M': 'M_HlthCare_SocAs_Tot', 'F': 'F_HlthCare_SocAs_Tot'},
    'Professional_Tech': {'M': 'M_Pro_scien_tec_Tot', 'F': 'F_Pro_scien_tec_Tot'}
}

# Merge gender-split tables
ind_df = pd.merge(ind_a_df, ind_b_df, on='SAL_CODE_2021')

# Calculate total persons per target industry
for industry, keys in industry_targets.items():
    ind_df[f'total_{industry.lower()}'] = ind_df[keys['M']] + ind_df[keys['F']]

# Aggregate all female industry totals to calculate total female workforce
f_tot_cols = [col for col in ind_b_df.columns if col.startswith('F_') and col.endswith('_Tot')]
ind_df['F_Total_Calculated'] = ind_df[f_tot_cols].sum(axis=1)

# Calculate total workforce (Male Total + Calculated Female Total)
ind_df['total_workforce'] = ind_df['M_Tot_Tot'] + ind_df['F_Total_Calculated']

# Validate output
display(ind_df[['SAL_CODE_2021', 'total_mining', 'total_education', 'total_workforce']].head())

,SAL_CODE_2021,total_mining,total_education,total_workforce
0,SAL50001,5,0,31
1,SAL50002,41,56,606
2,SAL50003,0,0,29
3,SAL50004,0,8,26
4,SAL50005,0,0,14


### 2.6 Process Labour Force Status (G46A & G46B)
To further understand the socio-economic drivers of housing stress, we extract labour force status data. Using the aggregated person totals from G46B, we extract counts for full-time employment, part-time employment, and unemployment to calculate future participation and unemployment rates.

In [214]:
# Create copy of G46B which contains the 'P' (Persons) aggregated totals
lfs_df = dfs['G46B'].copy()

# Define labour force categories for extraction using verified header names
# Use 'P' columns to get combined totals directly
lfs_targets = {
    'total_employed_full_time': 'P_Emp_FullT_Tot',
    'total_employed_part_time': 'P_Emp_PartT_Tot',
    'total_unemployed': 'P_Tot_Unemp_Tot',
    'total_labour_force': 'P_Tot_LF_Tot'
}

# Extract and rename columns for clarity
lfs_df = lfs_df[['SAL_CODE_2021'] + list(lfs_targets.values())]
lfs_df.rename(columns={v: k for k, v in lfs_targets.items()}, inplace=True)

# Validate output
display(lfs_df[['SAL_CODE_2021', 'total_employed_full_time', 'total_unemployed', 'total_labour_force']].head())

,SAL_CODE_2021,total_employed_full_time,total_unemployed,total_labour_force
0,SAL50001,25,0,51
1,SAL50002,335,17,640
2,SAL50003,19,0,46
3,SAL50004,17,0,28
4,SAL50005,11,0,18


### 2.7 Process Selected Characteristics (G01 & G02)
To normalize our industry and housing data into meaningful ratios, we extract total population and household counts. G01 provides the total person count, while G02 provides the average household size and median age, which serve as foundational denominators for the analysis.

In [215]:
# Create copies of G01 and G02
pop_df = dfs['G01'].copy()
avg_df = dfs['G02'].copy()

# Extract total population from G01
# Tot_P_P represents Total Persons
pop_df = pop_df[['SAL_CODE_2021', 'Tot_P_P']]
pop_df.rename(columns={'Tot_P_P': 'total_population'}, inplace=True)

# Extract average characteristics from G02
# Focus on median age and average household size
avg_cols = {
    'Median_age_persons': 'median_age',
    'Average_household_size': 'avg_household_size'
}
avg_df = avg_df[['SAL_CODE_2021'] + list(avg_cols.keys())]
avg_df.rename(columns=avg_cols, inplace=True)

# Merge characteristics into a single reference dataframe
char_df = pd.merge(pop_df, avg_df, on='SAL_CODE_2021')

# Validate output
display(char_df.head())

,SAL_CODE_2021,total_population,median_age,avg_household_size
0,SAL50001,83,39,3.0
1,SAL50002,1321,42,2.7
2,SAL50003,92,42,2.8
3,SAL50004,41,38,2.8
4,SAL50005,51,34,2.8


### 2.8 Process Dwelling Structure (G38)
To investigate the relationship between housing density and rental stress, we extract dwelling structure counts. We categorise these into 'Separate House' and 'High Density' (Apartments) using the aggregated totals provided in the G38 mortgage table to calculate density ratios for each suburb.

In [216]:
# Create copy of G38
structure_df = dfs['G38'].copy()

# Define columns for dwelling types based on verified G38 headers
# Tot_DS_Separate_hse: Separate houses
# Tot_DS_Flat_apartment: Flats, units, or apartments
structure_cols = {
    'Tot_DS_Separate_hse': 'count_separate_house',
    'Tot_DS_Flat_apartment': 'count_high_density'
}

# Extract and rename relevant columns using verified index keys
structure_df = structure_df[['SAL_CODE_2021'] + list(structure_cols.keys())]
structure_df.rename(columns=structure_cols, inplace=True)

# Calculate total dwellings monitored for density ratio
structure_df['total_dwellings_monitored'] = (
    structure_df['count_separate_house'] + 
    structure_df['count_high_density']
)

# Validate output
display(structure_df.head())

,SAL_CODE_2021,count_separate_house,count_high_density,total_dwellings_monitored
0,SAL50001,7,0,7
1,SAL50002,151,0,151
2,SAL50003,7,0,7
3,SAL50004,0,0,0
4,SAL50005,0,0,0


## 3. Data Integration and Feature Engineering
In this section, we consolidate the individual dataframes extracted from various Census tables into a master dataset. This process involves merging tabular data on the SAL_CODE_2021 key and engineering the specific ratios (Housing Stress, Industry Concentration, and Dwelling Density) required for our core investigation.

### 3.1 Data Integration
We first assemble the housing-specific variables into a single dataframe and then perform a sequential join with the industry, labour, and demographic datasets. This creates a unified 'master_df' containing all raw counts required for the analysis.

In [217]:
# Consolidate housing-specific data (Rent and Mortgage)
# rent_df (G40) and mortgage_df (G38) were created in Section 2
housing_df = pd.merge(rent_df, mortgage_df, on='SAL_CODE_2021', how='left')

# List of all processed DataFrames to merge
dfs_to_merge = [housing_df, ind_df, lfs_df, char_df, structure_df]

# Perform a sequential left join on SAL_CODE_2021
# Start with housing_df to preserve primary dependent variables
master_df = dfs_to_merge[0]

for df in dfs_to_merge[1:]:
    master_df = pd.merge(master_df, df, on='SAL_CODE_2021', how='left')

# Validate the final shape and column list
print(f"Master DataFrame Shape: {master_df.shape}")
print("Columns in Master DF:", master_df.columns.tolist())
display(master_df.head())

Master DataFrame Shape: (1701, 639)
Columns in Master DF: ['SAL_CODE_2021', 'R_1_74_LT_Real_estate_agent', 'R_1_74_LT_Ste_terr_hsg_auth', 'R_1_74_LT_com_hou_pro', 'R_1_74_LT_Psn_not_Sme_hhd', 'R_1_74_LT_Other_Lld_type', 'R_1_74_LT_Lld_type_ns', 'R_1_74_Tot', 'R_75_99_LT_Real_estate_agent', 'R_75_99_LT_Ste_terr_hsg_auth', 'R_75_99_LT_com_hou_pro', 'R_75_99_LT_Psn_not_Sme_hhd', 'R_75_99_LT_Other_Lld_type', 'R_75_99_LT_Lld_type_ns', 'R_75_99_Tot', 'R_100_149_LT_Real_eSte_agent', 'R_100_149_LT_St_ter_hsg_aut', 'R_100_149_LT_com_hou_pro', 'R_100_149_LT_Psn_not_Sme_hhd', 'R_100_149_LT_Other_Lld_type', 'R_100_149_LT_Lld_type_ns', 'R_100_149_Tot', 'R_150_199_LT_Real_eSte_agent', 'R_150_199_LT_Ste_ter_hsg_aut', 'R_150_199_LT_com_hou_pro', 'R_150_199_LT_Psn_not_Sme_hhd', 'R_150_199_LT_Other_Lld_type', 'R_150_199_LT_Lld_type_ns', 'R_150_199_Tot', 'R_200_224_LT_Real_eSte_agent', 'R_200_224_LT_Ste_ter_hsg_aut', 'R_200_224_LT_com_hou_pro', 'R_200_224_LT_Psn_not_Sme_hhd', 'R_200_224_LT_Other_Lld_type

,SAL_CODE_2021,R_1_74_LT_Real_estate_agent,R_1_74_LT_Ste_terr_hsg_auth,R_1_74_LT_com_hou_pro,R_1_74_LT_Psn_not_Sme_hhd,R_1_74_LT_Other_Lld_type,R_1_74_LT_Lld_type_ns,R_1_74_Tot,R_75_99_LT_Real_estate_agent,R_75_99_LT_Ste_terr_hsg_auth,...,total_employed_full_time,total_employed_part_time,total_unemployed,total_labour_force,total_population,median_age,avg_household_size,count_separate_house,count_high_density,total_dwellings_monitored
0,SAL50001,0,0,0,0,0,0,0,0,0,...,25,16,0,51,83,39,3.0,7,0,7
1,SAL50002,0,0,0,0,0,0,0,0,0,...,335,235,17,640,1321,42,2.7,151,0,151
2,SAL50003,0,0,0,0,0,0,0,0,0,...,19,18,0,46,92,42,2.8,7,0,7
3,SAL50004,0,0,0,0,0,0,0,0,0,...,17,5,0,28,41,38,2.8,0,0,0
4,SAL50005,0,0,0,0,0,0,0,0,0,...,11,4,0,18,51,34,2.8,0,0,0


**Observation:**
The `master_df` now serves as the unified source of truth, containing both our economic indicators (numerator data) and population benchmarks (denominator data).
* For example, **SAL50002** shows a `total_mining` count of **41** against a `total_workforce` of **606**, allowing us to calculate the specific industry concentration ratio.
* The presence of `total_population` and `avg_household_size` across all **1,701** rows ensures we can normalize housing costs against the actual scale of each suburb.
* High-density counts (like those in `count_high_density`) are now aligned with mortgage and rent averages, which is essential for testing our hypothesis regarding the impact of dwelling structure on housing stress.

This consolidated structure is the prerequisite for the **Feature Engineering** phase, where we transform these raw counts into standardized percentages and indices.

### 3.2 Feature Engineering - Ratios & Housing Stress Index
This step transforms raw counts into `standardised ratios`. We calculate `Employment Concentration` (Mining vs. Services), **Dwelling Density** (Apartments vs. Houses), and prepare the **Housing Stress Index**. 

We use the `NumPy` library to clean up our math. It helps us find and fix `infinity` errors or `missing values` that happen when we try to divide by zero in empty suburbs. Our main goal is to find areas where `housing costs` take up more than **30% of income**, making it easy to compare suburbs regardless of their size.

In [218]:
import numpy as np

# Calculate mining and service industry concentration ratios relative to workforce
master_df['mining_concentration_ratio'] = master_df['total_mining'] / master_df['total_workforce']
master_df['service_concentration_ratio'] = (master_df['total_healthcare'] + master_df['total_education']) / master_df['total_workforce']

# Determine unemployment rate using total labour force as denominator
master_df['unemployment_rate'] = master_df['total_unemployed'] / master_df['total_labour_force']

# Calculate high density ratio based on monitored dwelling structures
master_df['high_density_ratio'] = master_df['count_high_density'] / master_df['total_dwellings_monitored']

# Normalise monthly mortgage costs to a weekly scale for cost index comparison
master_df['avg_weekly_mortgage'] = master_df['avg_monthly_mortgage'] / 4.345

# Replace infinity and null values resulting from division by zero in empty suburbs
master_df.replace([np.inf, -np.inf], 0, inplace=True)
master_df.fillna(0, inplace=True)

# Define feature columns for final verification
feature_cols = [
    'SAL_CODE_2021', 'mining_concentration_ratio', 'service_concentration_ratio',
    'unemployment_rate', 'high_density_ratio', 'avg_weekly_rent', 'avg_weekly_mortgage'
]

# Validate output
display(master_df[feature_cols].head())

,SAL_CODE_2021,mining_concentration_ratio,service_concentration_ratio,unemployment_rate,high_density_ratio,avg_weekly_rent,avg_weekly_mortgage
0,SAL50001,0.161290,0.000000,0.000000,0.0,0.000000,698.005370
1,SAL50002,0.067657,0.249175,0.026562,0.0,369.956731,417.033867
2,SAL50003,0.000000,0.103448,0.000000,0.0,249.500000,474.568470
3,SAL50004,0.000000,0.307692,0.000000,0.0,0.000000,0.000000
4,SAL50005,0.000000,0.000000,0.000000,0.0,0.000000,0.000000


## 3.3 Data Validation and Quality Assurance
Before finalising the master dataset, we must perform a rigorous audit of our engineered features. In small-population areas, ABS confidentiality perturbation can occasionally result in logical anomalies (e.g. ratios exceeding 100%). We also examine financial outliers and "zero-density" suburbs to ensure our data is representative for future modelling.

### 3.3.1 Logical Bound Validation (Ratios)
We check our calculated ratio columns to ensure all values fall within the logical range of 0 to 1. This diagnostic helps identify where ABS data perturbation has created values that are mathematically impossible, such as a specific industry count exceeding the total workforce count.

In [219]:
# List ratio columns for validation
ratio_cols = ['mining_concentration_ratio', 'service_concentration_ratio', 
              'unemployment_rate', 'high_density_ratio']

# Check for values outside the expected 0 to 1 range
print("--- Ratio Logic Check ---")
for col in ratio_cols:
    outliers = master_df[(master_df[col] > 1.0) | (master_df[col] < 0)]
    if not outliers.empty:
        print(f"Anomaly found in {col}:")
        display(outliers[['SAL_CODE_2021', col, 'total_mining', 'total_workforce']])
    else:
        print(f"{col}: All values within logical bounds.")

--- Ratio Logic Check ---
Anomaly found in mining_concentration_ratio:


,SAL_CODE_2021,mining_concentration_ratio,total_mining,total_workforce
484,SAL50485,1.250000,10,8
664,SAL50665,1.142857,8,7


service_concentration_ratio: All values within logical bounds.
unemployment_rate: All values within logical bounds.
high_density_ratio: All values within logical bounds.


#### Observation and Fix: Ratio Anomalies
The diagnostic identified two suburbs where the `mining_concentration_ratio` exceeded **100%**. This is an **artifact of ABS perturbation** where small counts ($N < 10$) are randomly adjusted for privacy. Because this affects only a tiny fraction of the dataset and represents a minor **mathematical overshoot**, we use `.clip()` to cap these values at `1.0`. This maintains **logical consistency** for future modelling without losing the high-concentration signal.

In [220]:
# Cap ratios at 1.0 to handle perturbation anomalies
master_df[ratio_cols] = master_df[ratio_cols].clip(0, 1)
print("Fix applied: All ratios capped at 1.0.")

Fix applied: All ratios capped at 1.0.


### 3.3.2 Financial and Zero-Density Audit
We investigate the presence of 'zero-density' suburbs, such as industrial areas or national parks, which may have no recorded workforce. We also audit our financial columns for outliers or unexpected zero-dollar values in rent and mortgage data to ensure the dataset is representative of residential housing.

In [221]:
# Check for suburbs with no workforce
zero_workforce = master_df[master_df['total_workforce'] == 0]
print(f"Suburbs with zero workforce: {len(zero_workforce)}")

# Check for suburbs with workforce but no recorded rent
zero_rent_with_pop = master_df[(master_df['avg_weekly_rent'] == 0) & (master_df['total_workforce'] > 0)]
print(f"Suburbs with workforce but $0 avg rent: {len(zero_rent_with_pop)}")

# Check for high rent outliers above $2000 per week
high_rent = master_df[master_df['avg_weekly_rent'] > 2000]
print(f"High-rent outliers found: {len(high_rent)}")

# Show final validation summary
print("\nValidation Summary:")
print(f"Total Rows: {len(master_df)}")
print(f"Suburbs with valid workforce data: {len(master_df) - len(zero_workforce)}")

Suburbs with zero workforce: 175
Suburbs with workforce but $0 avg rent: 645
High-rent outliers found: 0

Validation Summary:
Total Rows: 1701
Suburbs with valid workforce data: 1526


#### Observation: Zero-Density and Financials
We identified **175 suburbs** with `zero workforce`, which typically represent **industrial zones**, **national parks**, or uninhabited areas. Additionally, **645 suburbs** show `$0` `avg_weekly_rent` despite having residents; this is a common characteristic of **regional Western Australia** where housing is often **employer-provided** (mining/agriculture) or owned outright. No surgical fix is required for these values as they accurately reflect the **regional economic profile**.

## 4. Geospatial Integration
This section focuses on connecting our statistical results to physical locations. We will combine our master dataset with geographic boundary files to enable mapping and spatial analysis.

### 4.1 Geospatial Join & Validation
We introduce the `GeoPandas` library here to manage geographic data. It allows us to load a `Shapefile`, which contains the actual map coordinates for Western Australian suburbs. By joining this with our `master_df`, we create a `GeoDataFrame` that links our housing stress calculations to specific spots on the map.

In [222]:
import geopandas as gpd

# Load the national shapefile from the spatial subdirectory
shapefile_path = '../data/spatial/SAL_2021_AUST_GDA2020.shp'
wa_gdf = gpd.read_file(shapefile_path)

# Filter for Western Australia and rename the key column
wa_gdf = wa_gdf[wa_gdf['SAL_CODE21'].str.startswith('5')].copy()
wa_gdf = wa_gdf.rename(columns={'SAL_CODE21': 'SAL_CODE_2021'})

# Strip the 'SAL' prefix from the master_df codes to match shapefile IDs
master_df['SAL_CODE_2021'] = master_df['SAL_CODE_2021'].str.replace('SAL', '', regex=False)

# Perform the inner join to create the spatial dataset
final_geo_df = wa_gdf.merge(master_df, on='SAL_CODE_2021', how='inner')

# Validate the join results
print(f"Shapes found for WA: {wa_gdf.shape[0]}")
print(f"Rows in master_df: {master_df.shape[0]}")
print(f"Final merged GeoDataFrame rows: {final_geo_df.shape[0]}")

# Validate spatial data output
display(final_geo_df[['SAL_NAME21', 'geometry', 'avg_weekly_rent']].head())

Shapes found for WA: 1701
Rows in master_df: 1701
Final merged GeoDataFrame rows: 1701


,SAL_NAME21,geometry,avg_weekly_rent
0,Abba River,"POLYGON ((115.53341 -33.69941, 115.52549 -33.6...",0.000000
1,Abbey,"POLYGON ((115.26517 -33.66805, 115.26496 -33.6...",369.956731
2,Acton Park (WA),"POLYGON ((115.41094 -33.76421, 115.41171 -33.7...",249.500000
3,Adamsvale,"POLYGON ((117.85683 -32.22837, 117.85672 -32.2...",0.000000
4,Ajana,"POLYGON ((114.67557 -27.72687, 114.71635 -27.7...",0.000000


### 4.2 Spatial Data Observation
We perform a brief check of the merged data. We want to ensure that the geographic boundaries align with our expectation of Western Australian suburbs and that our calculated columns (like rent and mortgage) are correctly attached to their physical polygons.

In [223]:
# Check for null values in the geometry or key columns
print(f"Suburbs with missing geometry: {final_geo_df['geometry'].isna().sum()}")
print(f"Suburbs with missing rent data: {final_geo_df['avg_weekly_rent'].isna().sum()}")

# Basic spatial summary
print("\nSpatial Extent (Bounding Box):")
print(final_geo_df.total_bounds)

# Check of top 5 suburbs by mining concentration
display(final_geo_df.nlargest(5, 'mining_concentration_ratio')[['SAL_NAME21', 'mining_concentration_ratio', 'geometry']])

Suburbs with missing geometry: 2
Suburbs with missing rent data: 0

Spatial Extent (Bounding Box):
[112.92112452 -35.19199451 129.0019278  -13.68869015]


,SAL_NAME21,mining_concentration_ratio,geometry
127,Binduli,1.0,"POLYGON ((121.40575 -30.78051, 121.4084 -30.78..."
484,Emu Flat (WA),1.0,"POLYGON ((121.68397 -30.86114, 121.68593 -30.8..."
664,Indee,1.0,"POLYGON ((118.48476 -20.70415, 118.52802 -20.7..."
734,Karramindie,1.0,"POLYGON ((121.56025 -31.17502, 121.56025 -31.1..."
800,Kurnalpi,1.0,"POLYGON ((122.26746 -30.60637, 122.2561 -30.60..."


#### Observation: Spatial Join and Missing Geometry
The spatial join successfully matched the majority of the Census dataset to the Western Australian suburb boundaries. A check of the top 5 suburbs by `mining_concentration_ratio` confirms that the highest concentrations (100%) are correctly mapped to known remote mining localities like **Binduli** and **Indee**.

However, the diagnostic identified **2 rows** with missing geometry. Given that the bounding box and overall alignment are correct, these missing rows require specific investigation to ensure no critical spatial data was lost.

### 4.3 Handling Non-Spatial "Ghost" Categories

To keep our maps accurate, we need to handle the suburbs that did not find a matching shape in the previous step. Instead of assuming these are errors, we will identify the **SAL codes** missing geometry and check them against our `master_df`.

This helps us verify if these are specific ABS categories (like **'Migratory / Offshore / Shipping'**) and see how many workers they represent before we remove them from the mapping dataset.

In [224]:
# Identify codes with no geometry
null_mask = final_geo_df['geometry'].isna()
missing_codes = final_geo_df.loc[null_mask, 'SAL_CODE_2021'].unique().tolist()

# Fallback if re-running this cell
if not missing_codes:
    missing_codes = ['59494', '59797']

# Get data from master dataframe
missing_data = master_df[master_df['SAL_CODE_2021'].astype(str).str.contains('|'.join(missing_codes))]

# Calculate workforce total
workforce_total = missing_data['total_workforce'].sum()

print(f"Codes identified: {missing_codes}")
print(f"Total workforce: {workforce_total:,.0f}")

# Display verification details
display(missing_data[['SAL_CODE_2021', 'total_workforce', 'mining_concentration_ratio']])

# Filter for valid geometry
final_geo_df = final_geo_df[final_geo_df['geometry'].notna()].copy()

print(f"\nFinal suburb count: {len(final_geo_df)}")

Codes identified: ['59494', '59797']
Total workforce: 1,855


,SAL_CODE_2021,total_workforce,mining_concentration_ratio
1699,59494,1592,0.121231
1700,59797,263,0.307985



Final suburb count: 1699


#### Observation: Non-Spatial Workforce Analysis
The identified codes (**59494** and **59797**) are the standard administrative markers used by the ABS for **'Migratory / Offshore / Shipping'** and **'No usual address'**. This verification explains why they failed the spatial join: they represent mobile or non-localized populations rather than physical suburbs.

These rows account for a combined workforce of **1,855 individuals**. While this reflects the **FIFO and offshore** nature of the Western Australian mining industry, these records cannot be represented on our map. Removing them ensures `final_geo_df` is clean for spatial visualization while maintaining an audit trail of why these specific workers were excluded.